# Парсинг сайтов 

In [1]:
from bs4 import BeautifulSoup as bs

In [2]:
import pandas as pd

In [3]:
from curl_cffi import requests

In [4]:
import time

In [5]:
import random

In [6]:
import re

In [7]:
import os

# Получение информации

In [8]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Linux; Android 13; SM-G991B) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.6099.144 Mobile Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'ru-RU,ru;q=0.9,en;q=0.8',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive',
    'Sec-Fetch-Site': 'same-origin',
}

In [9]:
url = 'https://om.kinogo-filmov.net/top250/'

In [10]:
page = requests.get(url, headers=headers, verify=False, timeout=30, impersonate="chrome120")

In [11]:
page.status_code

403

In [12]:
soup = bs(page.text, 'html.parser')

In [13]:
page.text

'<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scale=1"><meta http-equiv="content-security-policy" content="default-src &#39;none&#39;; script-src &#39;nonce-hsys4TutxGztqTecN6U8oC&#39; &#39;unsafe-eval&#39; https://challenges.cloudflare.com; script-src-attr &#39;none&#39;; style-src &#39;unsafe-inline&#39;; img-src &#39;self&#39; https://challenges.cloudflare.com; connect-src &#39;self&#39; https://challenges.cloudflare.com; frame-src &#39;self&#39; https://challenges.cloudflare.com blob:; child-src &#39;self&#39; https://challenges.cloudflare.com blob:; worker-src blob:; form-action http: https:; base-uri &#39;self&#39;"><style>*{box-sizing:border-box;margin:0;padding:0}html{line-height:1.15;-webkit-text-size-adjust:100%;color:#313131;fon

In [14]:
soup

<!DOCTYPE html>
<html lang="en-US"><head><title>Just a moment...</title><meta content="text/html; charset=utf-8" http-equiv="Content-Type"/><meta content="IE=Edge" http-equiv="X-UA-Compatible"/><meta content="noindex,nofollow" name="robots"/><meta content="width=device-width,initial-scale=1" name="viewport"/><meta content="default-src 'none'; script-src 'nonce-hsys4TutxGztqTecN6U8oC' 'unsafe-eval' https://challenges.cloudflare.com; script-src-attr 'none'; style-src 'unsafe-inline'; img-src 'self' https://challenges.cloudflare.com; connect-src 'self' https://challenges.cloudflare.com; frame-src 'self' https://challenges.cloudflare.com blob:; child-src 'self' https://challenges.cloudflare.com blob:; worker-src blob:; form-action http: https:; base-uri 'self'" http-equiv="content-security-policy"/><style>*{box-sizing:border-box;margin:0;padding:0}html{line-height:1.15;-webkit-text-size-adjust:100%;color:#313131;font-family:system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,"Helv

In [15]:
result_list = {'title': [], 'description': [], 'date': [], 'views': []}

# Алгоритм

In [16]:
def clean_text(text):
    """Очищает текст от лишних пробелов и переносов"""
    if text:
        return re.sub(r'\s+', ' ', text).strip()
    return None

In [17]:
def parse_movie_regex(html_text):
    """Парсит один фильм через регулярные выражения"""
    
    movie = {}
    
    # Название 
    title_match = re.search(r'<h2 class="zagolovki"><a href="([^"]+)">([^<]+)</a></h2>', html_text)
    if title_match:
        title = title_match.group(2)
        title = re.sub(r'\s*\(\d{4}\)\s*$', '', title)
        movie['name'] = title.strip()
    
    # Год выпуска
    year_match = re.search(r'<b>Год выпуска:</b>\s*<a[^>]*>(\d{4})</a>', html_text)
    movie['year'] = year_match.group(1) if year_match else None

    # Страна
    desc_match = re.search(r'<!--TEnd-->\s*([^<]+(?:<[^>]+>[^<]*)?)', html_text)
    if desc_match:
        description = desc_match.group(1)
        # Убираем "...<br><br>" и лишние пробелы
        description = re.sub(r'\s+', ' ', description).strip()
        # Обрезаем до 300 символов
        movie['description'] = description[:300] + '...' if len(description) > 300 else description
    else:
        movie['description'] = None
    
    # Страна (Выпущено)
    country_match = re.search(r'<b>Выпущено:</b>\s*(.+?)(?=<b>|$)', html_text, re.DOTALL)
    if country_match:
        country_block = country_match.group(1)
        countries = re.findall(r'<a[^>]*>([^<]+)</a>', country_block)
        if countries:
            movie['countries'] = ', '.join(countries)
        else:
            countries_text = re.sub(r'<[^>]+>', '', country_block).strip()
            movie['countries'] = countries_text if countries_text else None
    else:
        movie['countries'] = None
    
        
    # Рейтинг IMDB
    imdb_match = re.search(r'<b>Рейтинг IMDB:</b>\s*([\d.]+)', html_text)
    movie['rating'] = imdb_match.group(1) if imdb_match else None

    # Жанр
    genre_match = re.search(r'<b>Жанр:</b>\s*(.+?)(?=<b>|$)', html_text, re.DOTALL)
    if genre_match:
        genre_block = genre_match.group(1)
        genres = re.findall(r'<a[^>]*>([^<]+)</a>', genre_block)
        if genres:
            movie['genres'] = ', '.join(genres)
        else:
            genres_text = re.sub(r'<[^>]+>', '', genre_block).strip()
            movie['genres'] = genres_text if genres_text else None
    else:
        movie['genres'] = None
    
    return movie

In [18]:
numPage = 1 
all_movies = []
# https://om.kinogo-filmov.net/top250/page/
for i in range(18):
    page_url = f"https://m.kinogo.online/page/{numPage}"
    time.sleep(random.uniform(15,30))
    iter_page = requests.get(page_url, headers=headers, verify=False, timeout=10)
    
    if iter_page.status_code != 200:
        print(f'Ошибка: {iter_page.status_code}')
        continue
        
    iter_soup = bs(iter_page.text, 'html.parser')
    items = iter_soup.find_all("div", class_="shortstory")
    
    for item in items:
        movie_html = str(item)
        movie_data = parse_movie_regex(movie_html)
        if (movie_data.get('name')):
            all_movies.append(movie_data)
            print(f'фильм {movie_data.get('name')}')
            
    print(len(items))
    numPage += 1
print('!!!!!Парс окончен')

KeyboardInterrupt: 

In [ ]:
df = pd.DataFrame(all_movies)
df
df = df.head(500)
df

In [ ]:
df.to_csv('films.csv', index=False, mode="a", header=not (os.path.isfile('films.csv')))